# 基底変換とTT-rank不変性: U ⊗ I の小規模検証

このNotebookでは、添付チャットで確認した

\[
X^{\langle2\rangle}
=
(U\otimes I_{n_2})B^{\langle2\rangle}
\]

と、

\[
\operatorname{rank}(X^{\langle2\rangle})
=
\operatorname{rank}(B^{\langle2\rangle})
\]

を小さい3階テンソルで検証する。

数値誤差を見やすくするため、このNotebookでも `float64` を使う。

## ゴール

- 第1回SVD後の `U` が列直交であることを確認する
- $U\otimes I$ のshapeを追う
- 第2切断で上の等式を数値的に確認する
- 基底変換だけならrankが変わらないことを確認する
- truncation後は、近似テンソル $\hat X$ について同じrank不変性が残ることを確認する
- 元の $X$ と近似 $\hat X$ のrankは一般に同じとは限らないことを確認する

> gauge freedomはまだ扱わない。

## 1. 3階テンソルと第1回SVD

$X\in\mathbb{R}^{2\times3\times4}$ を用意し、
第1切断 $i_1 \mid i_2i_3$ でSVDする。

### やること

- `X1.shape == (2, 12)`
- SVD後、`r1 = matrix_rank(X1)` を求める
- `U`, `S`, `Vh` を `r1` まで切る
- `B = Sigma @ Vh`
- `B` を `(r1, n2, n3)` と読めるshapeへ戻す

> `torch.linalg.matrix_rank` はしきい値に基づく数値rankを返す。今回のランダムな小テンソルでは問題になりにくいが、ほぼゼロの特異値を持つテンソルでは tolerance の選び方が結果に影響する。

In [ ]:
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(3)

# TODO: X.shape=(2, 3, 4)
X = None

# TODO: 第1切断で X1.shape=(2, 12)
X1 = None

# TODO: SVD
U = None
S = None
Vh = None

# TODO: r1 = matrix_rank(X1) とし、U / S / Vh を r1 まで切る
r1 = None

# TODO: B = Sigma @ Vh
B = None

## 2. Uの列直交性

数値rankまで切った `U` も列直交なので

\[
U^\top U=I_{r_1}
\]

を満たす。

### やること

- `U.T @ U`
- 単位行列との差のFrobenius norm

を確認する。

### ヒント

- `torch.eye`
- `torch.linalg.matrix_norm(..., ord="fro")`

`float64` なら、直交性誤差が $10^{-14}$〜$10^{-15}$ 程度でも浮動小数点の数値誤差として自然。

In [ ]:
# TODO: U.T @ U と I の誤差を計算する
orth_error = None

## 3. 第2切断のXとB

元テンソルの第2切断は $(i_1i_2)\mid i_3$ なので

\[
X^{\langle2\rangle}
\in\mathbb{R}^{(n_1n_2)\times n_3}
\]

となる。

一方、SVD後の残りは $(\alpha_1i_2)\mid i_3$ なので

\[
B^{\langle2\rangle}
\in\mathbb{R}^{(r_1n_2)\times n_3}
\]

となる。

### やること

- `X_cut2`
- `B_cut2`

をそれぞれreshapeしてshapeを確認する。

In [ ]:
# TODO: X_cut2.shape = (n1*n2, n3)
X_cut2 = None

# TODO: B_cut2.shape = (r1*n2, n3)
B_cut2 = None

## 4. U ⊗ I を作る

\[
L_2=U\otimes I_{n_2}
\]

を作る。

これは「第1mode側には基底変換 `U` を作用させ、
第2modeには何もしない」変換に対応する。

### やること

- `L2 = U ⊗ I`
- shapeを確認する
- `L2.T @ L2 = I` を確認する

### ヒント

- `torch.kron`
- `torch.eye(n2)`

In [ ]:
# TODO: L2 = U kron I_n2
L2 = None

# TODO: L2の列直交性も確認する
l2_orth_error = None

## 5. X^(2) = (U ⊗ I) B^(2) を確認

### やること

\[
X^{\langle2\rangle}
\stackrel{?}{=}
L_2 B^{\langle2\rangle}
\]

を数値的に確認する。

さらに

```python
torch.linalg.matrix_rank(X_cut2)
torch.linalg.matrix_rank(B_cut2)
```

を比較する。

### 見る点

`L2` が列フルランクで左逆を持つため、
基底変換だけでは独立な方向を潰さない。

In [ ]:
# TODO: transformed = L2 @ B_cut2
transformed = None

# TODO: X_cut2との差
relation_error = None

# TODO: rankを比較する
rank_x = None
rank_b = None

## 6. truncationとの違い

ここまでは第1回SVDで数値的に非零な特異方向をすべて残している。

次に第1回SVDを意図的にrank $\hat r_1<r_1$ へ打ち切り、近似テンソル $\hat X$ を作る。
そのとき、打ち切った `U`, `S`, `Vh` から $\hat B$ と $\hat L_2$ を作る。

重要なのは、**打ち切り後にも近似テンソル同士では基底変換によるrank不変性が成り立つ**こと。

\[
\hat X^{\langle2\rangle}
=
\hat L_2\hat B^{\langle2\rangle}
\]

であり、$\hat L_2$ が列フルランクなら

\[
\operatorname{rank}(\hat X^{\langle2\rangle})
=
\operatorname{rank}(\hat B^{\langle2\rangle})
\]

は依然として成り立つ。

ただし比較対象を元テンソルへ戻すと、一般には

\[
\operatorname{rank}(\hat X^{\langle2\rangle})
\neq
\operatorname{rank}(X^{\langle2\rangle})
\]

となり得る。

### やること

1. 第1回SVDを意図的にtruncateする
2. `U_hat`, `S_hat`, `Vh_hat` から `B_hat` と `L2_hat` を作る
3. $\hat X$ を再構成する
4. `rank(X_hat_cut2) == rank(B_hat_cut2)` を確認する
5. `rank(X_hat_cut2)` と元の `rank(X_cut2)` を比較する

### 確認したいこと

\[
\boxed{\text{基底変換だけなら、その表現対象についてrankは不変}}
\]

一方で

\[
\boxed{\text{truncationで表現対象そのものを }X\to\hat X\text{ に変えるとrankは変わり得る}}
\]

という二段階を区別する。

In [ ]:
# TODO: 第1SVDを意図的にtruncateする

# TODO: U_hat / B_hat / L2_hat を作る

# TODO: X_hat を再構成し、第2切断 X_hat_cut2 を作る

# TODO: rank(X_hat_cut2) == rank(B_hat_cut2) を確認する

# TODO: rank(X_hat_cut2) と元の rank(X_cut2) を比較する

## 7. ここまでの確認

この4冊で、添付チャットでgauge freedomより前に学習した範囲を
小規模実験として一通り検証したことになる。

1. TT定義・3階TT-SVD・完全再構成
2. TT-rank = cut unfolding rank
3. truncationによるrank / 保存量 / 誤差のtrade-off
4. $U\otimes I$ による基底変換とrank不変性
5. truncation後は $\hat X$ と $\hat B$ のrank不変性と、元の $X$ との差を区別する

次に理論で gauge freedom を学んだ後、
その内容に対応する新しいNotebookを追加する。